# Transformer Summarizer — Abstractive Russian News Summarization

Train a Transformer encoder-decoder to generate abstractive summaries of
Russian news articles, on the
[IlyaGusev/gazeta](https://huggingface.co/datasets/IlyaGusev/gazeta) corpus.

## Setup

In [38]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
import sentencepiece as spm
import torch
from colorama import Fore, Style
from datasets import load_dataset
from dotenv import load_dotenv
from torch.utils.data import DataLoader
from torchmetrics.text import BLEUScore

from dl_roadmap.chapters.summarization import (
    SummarizationDataset,
    Summarizer,
    prepare_gazeta,
)
from dl_roadmap.chapters.summarization.dataset import make_collate_fn
from dl_roadmap.engine import (
    CombinedEarlyStopping,
    GapThresholdEarlyStopping,
    TeacherForcingTrainer,
    TrainerConfig,
    ValLossEarlyStopping,
    make_token_loss,
)
from dl_roadmap.utils import (
    LoggerConfig,
    load_model,
    notify_model_trained,
    save_model,
    seed_everything,
    setup_logger,
)
from dl_roadmap.visualization import plot_training_history

In [40]:
%matplotlib inline

pd.set_option("display.width", 150)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)

load_dotenv()
seed_everything()
setup_logger(LoggerConfig(log_level="WARNING"))

In [41]:
PROJECT_NAME = "10_summarizer"

DATA_DIR = Path("../data")
MODEL_DIR = Path("../models") / PROJECT_NAME
REPORT_DIR = Path("../reports")

RAW_DATA_DIR = DATA_DIR / "raw" / PROJECT_NAME
PROCESSED_DATA_DIR = DATA_DIR / "processed" / PROJECT_NAME
TOKENIZER_DIR = MODEL_DIR / "tokenizer"
OPTUNA_DIR = MODEL_DIR / "optuna"
FIG_DIR = REPORT_DIR / "figures" / PROJECT_NAME


DIRECTORIES = (
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    MODEL_DIR,
    TOKENIZER_DIR,
    OPTUNA_DIR,
    FIG_DIR,
)

for directory in DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

## Dataset

In [42]:
dataset = load_dataset("IlyaGusev/gazeta", cache_dir=RAW_DATA_DIR)

train_df = dataset["train"].shuffle(seed=42).to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

### Overview

In [43]:
print(f"{Fore.MAGENTA}DataFrame Info:{Style.RESET_ALL}")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{Fore.CYAN}====== {name} ======{Style.RESET_ALL}")
    df.info()

DataFrame Info:

====== train ======
<class 'pandas.DataFrame'>
RangeIndex: 60964 entries, 0 to 60963
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   text     60964 non-null  str  
 1   summary  60964 non-null  str  
 2   title    60964 non-null  str  
 3   date     60964 non-null  str  
 4   url      60964 non-null  str  
dtypes: str(5)
memory usage: 522.9 MB

====== val ======
<class 'pandas.DataFrame'>
RangeIndex: 6369 entries, 0 to 6368
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   text     6369 non-null   str  
 1   summary  6369 non-null   str  
 2   title    6369 non-null   str  
 3   date     6369 non-null   str  
 4   url      6369 non-null   str  
dtypes: str(5)
memory usage: 53.3 MB

====== test ======
<class 'pandas.DataFrame'>
RangeIndex: 6793 entries, 0 to 6792
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------

In [44]:
print(f"{Fore.YELLOW}First Rows of DataFrame:{Style.RESET_ALL}")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{Fore.CYAN}====== {name} ======{Style.RESET_ALL}")
    print(df.head())

First Rows of DataFrame:

====== train ======
                                                text                                            summary                                   title  \
0  В сети появилось видео задержания 58-летнего в...  Департамент транспорта Москвы проведет внеплан...  «Мы проведем тщательное расследование»   
1  В преддверии первого матча раунда плей-офф Лиг...  Накануне матча с «Рубином» главный тренер «Лио...          «Рубин» не боимся, но уважаем»   
2  Mail.ru Group , владеющая 39,99% «ВКонтакте», ...  Павел Дуров получит операционный контроль над ...               «ВКонтакте» под контролем   
3  Американский сенатор Джон Маккейн отреагировал...  Сенатор-республиканец Джон Маккейн молится за ...      Танкер протаранил «Джона Маккейна»   
4  Во вторник норвежский Нобелевский комитет расп...  Вручение Нобелевской премии мира спровоцировал...                Нобелевская премия войны   

                  date                                                u

In [45]:
print(f"{Fore.RED}Missing Values in Each Column:{Style.RESET_ALL}")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{Fore.CYAN}====== {name} ======{Style.RESET_ALL}")
    missing = df.isnull().sum()
    missing_percent = (missing / len(df)) * 100
    missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_percent})
    print(missing_df)

Missing Values in Each Column:

====== train ======
         Missing Count  Missing %
text                 0        0.0
summary              0        0.0
title                0        0.0
date                 0        0.0
url                  0        0.0

====== val ======
         Missing Count  Missing %
text                 0        0.0
summary              0        0.0
title                0        0.0
date                 0        0.0
url                  0        0.0

====== test ======
         Missing Count  Missing %
text                 0        0.0
summary              0        0.0
title                0        0.0
date                 0        0.0
url                  0        0.0


### Preprocessing

In [ ]:
train_df = prepare_gazeta(train_df, cache_file=PROCESSED_DATA_DIR / "train_df.csv")
val_df = prepare_gazeta(val_df, cache_file=PROCESSED_DATA_DIR / "val_df.csv")
test_df = prepare_gazeta(test_df, cache_file=PROCESSED_DATA_DIR / "test_df.csv")

In [47]:
print(f"{Fore.MAGENTA}DataFrame Info:{Style.RESET_ALL}")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{Fore.CYAN}====== {name} ======{Style.RESET_ALL}")
    df.info()

DataFrame Info:

====== train ======
<class 'pandas.DataFrame'>
RangeIndex: 60421 entries, 0 to 60420
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   text     60421 non-null  str  
 1   summary  60421 non-null  str  
dtypes: str(2)
memory usage: 506.1 MB

====== val ======
<class 'pandas.DataFrame'>
RangeIndex: 6321 entries, 0 to 6320
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   text     6321 non-null   str  
 1   summary  6321 non-null   str  
dtypes: str(2)
memory usage: 51.3 MB

====== test ======
<class 'pandas.DataFrame'>
RangeIndex: 6741 entries, 0 to 6740
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   text     6741 non-null   str  
 1   summary  6741 non-null   str  
dtypes: str(2)
memory usage: 55.9 MB


In [48]:
print(f"{Fore.YELLOW}First Rows of DataFrame:{Style.RESET_ALL}")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{Fore.CYAN}====== {name} ======{Style.RESET_ALL}")
    print(df.head())

First Rows of DataFrame:

====== train ======
                                                text                                            summary
0  В сети появилось видео задержания 58-летнего в...  Департамент транспорта Москвы проведет внеплан...
1  В преддверии первого матча раунда плей-офф Лиг...  Накануне матча с «Рубином» главный тренер «Лио...
2  Mail.ru Group , владеющая 39,99% «ВКонтакте», ...  Павел Дуров получит операционный контроль над ...
3  Американский сенатор Джон Маккейн отреагировал...  Сенатор-республиканец Джон Маккейн молится за ...
4  Во вторник норвежский Нобелевский комитет расп...  Вручение Нобелевской премии мира спровоцировал...

====== val ======
                                                text                                            summary
0  В 2020 году инфляция в России составит 3,5-4%,...  В уходящем году инфляция в России находится на...
1  Глава Белого дома Дональд Трамп выразил надежд...  Мировая общественность призвала лидера КНДР Ки...

### SentencePiece

In [49]:
force_tokenizer_train = False

if force_tokenizer_train or not (TOKENIZER_DIR / "sp.model").is_file():
    data_path = PROCESSED_DATA_DIR / "data.txt"
    data_path.write_text("\n".join(train_df["text"]), encoding="utf-8")

    spm.SentencePieceTrainer.train(
        input=str(data_path),
        model_prefix=str(TOKENIZER_DIR / "sp"),
        vocab_size=8000,
        model_type="unigram",
        max_sentence_length=16384,
        num_threads=os.cpu_count() or 4,
        pad_id=3,
        unk_id=2,
        bos_id=0,
        eos_id=1,
        pad_piece="<PAD>",
        unk_piece="<UNK>",
        bos_piece="<BOS>",
        eos_piece="<EOS>",
    )

In [50]:
sp = spm.SentencePieceProcessor(model_file=str(TOKENIZER_DIR / "sp.model"))

### Collation

In [ ]:
pad_id = sp.pad_id()
collate_fn = make_collate_fn(pad_id)

## Tools

In [ ]:
loss = make_token_loss(pad_id, label_smoothing=0.1)

## Training

In [55]:
dropout = 0.2
weight_decay = 1e-4

In [56]:
seed = 42

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [57]:
train_dataset = SummarizationDataset(train_df, sp)
val_dataset = SummarizationDataset(val_df, sp)
test_dataset = SummarizationDataset(test_df, sp)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
)

In [ ]:
model = Summarizer(
    sp,
    model_dim=128,
    num_heads=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    ffn_dim=(2 * 128),
    dropout=5e-2,
)

decay, no_decay = [], []
for n, p in model.named_parameters():
    if n.endswith("bias") or "norm" in n:
        no_decay.append(p)
    else:
        decay.append(p)

opt = torch.optim.AdamW(
    [
        {"params": decay, "weight_decay": 1e-2},
        {"params": no_decay, "weight_decay": 0.0},
    ],
    lr=3e-4,
)

warmup_steps = 2000
warmup = torch.optim.lr_scheduler.LinearLR(
    opt, start_factor=1 / warmup_steps, total_iters=warmup_steps - 1
)


def warmup_step(*_) -> None:
    """Advances the LR ramp once per optimizer step, then leaves the LR alone."""
    if warmup.last_epoch < warmup.total_iters:
        warmup.step()


opt.register_step_post_hook(warmup_step)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt,
    mode="min",
    factor=0.3,
    patience=3,
    min_lr=3e-5,
    threshold=0.01,
    threshold_mode="abs",
    cooldown=1,
)

val_stopping = ValLossEarlyStopping(patience=5, min_delta=1e-4)
early_stopping = CombinedEarlyStopping(
    [val_stopping, GapThresholdEarlyStopping(patience=5, threshold=0.8)], combine="all"
)

trainer_config = TrainerConfig(
    epochs=100,
    restore_best_weights=True,
    grad_clip_norm=1.0,
    grad_normalizer=loss.grad_normalizer,
)
trainer = TeacherForcingTrainer(
    model=model,
    optimizer=opt,
    loss_fn=loss.loss_fn,
    scheduler=scheduler,
    config=trainer_config,
    early_stopping=early_stopping,
    loss_tracker=loss.loss_tracker,
)

In [ ]:
train_model = False
model_path = MODEL_DIR / f"summarizer-{datetime.now():%Y.%m.%d_%H:%M:%S}.model"


def train() -> None:
    """Fits the model, saves the checkpoint, and reports the final losses."""
    trainer.fit(train_loader, val_loader)
    save_model(model, model_path)
    notify_model_trained(
        "Summarizer",
        {
            "best_epoch": val_stopping.best_epoch,
            "train_loss": trainer.history["train_loss"][-1],
            "val_loss": trainer.history["val_loss"][-1],
        },
    )


if train_model or not model_path.is_file():
    train()
else:
    load_model(model, model_path)

## Results

In [60]:
plot_training_history(
    **trainer.history,
    best_epoch=val_stopping.best_epoch,
    filename=FIG_DIR / "training_history.png",
)

---

In [ ]:
@torch.no_grad()
def generate_summary(
    text: str,
    model: Summarizer,
    sp: spm.SentencePieceProcessor,
) -> str:
    """Decodes a summary for an article using beam search.

    Args:
        text: Raw article text to summarize.
        model: A trained ``Summarizer`` model.
        sp: SentencePiece model used to encode ``text`` and decode the
            generated ids.

    Returns:
        The generated summary text, decoded up to (and excluding) ``<EOS>``.
    """
    device = next(model.parameters()).device

    src_ids = [sp.bos_id(), *sp.encode(text, out_type=int), sp.eos_id()]
    if len(src_ids) > SummarizationDataset.MAX_TEXT_LEN:
        src_ids = [*src_ids[: SummarizationDataset.MAX_TEXT_LEN - 1], sp.eos_id()]

    x = torch.tensor([src_ids], dtype=torch.long, device=device)

    ids = model.generate(x, max_length=SummarizationDataset.MAX_SUMMARY_LEN)[0].tolist()
    return sp.decode(ids)

---

In [71]:
for _, row in val_df.head(3).iterrows():
    summary = generate_summary(row["text"], model, sp)

    print(f"{Fore.CYAN}Text:{Style.RESET_ALL} {row['text'][:300]}...")
    print(f"{Fore.GREEN}Reference:{Style.RESET_ALL} {row['summary']}")
    print(f"{Fore.YELLOW}Generated:{Style.RESET_ALL} {summary}\n")

2038
1024
[0, 28, 3003, 116, 7284, 7, 78, 2906, 285, 4, 214, 21, 1795, 4, 1266, 1226, 6045, 5, 28, 1054, 4561, 206, 3495, 130, 9, 2681, 2512, 6, 1730, 5, 905, 4, 11, 843, 64, 3686, 401, 1101, 4795, 2143, 272, 5249, 3269, 185, 6238, 14, 3167, 73, 5, 10, 1130, 5, 215, 22, 79, 2050, 549, 4, 1915, 6, 2417, 19, 72, 2579, 497, 7, 3426, 116, 3269, 5331, 605, 5, 28, 1085, 116, 1921, 3548, 140, 8, 21, 1114, 1685, 46, 5450, 11, 6, 2356, 9, 4, 50, 2584, 4, 3359, 428, 464, 8, 11, 525, 283, 7477, 52, 2847, 2488, 684, 382, 119, 159, 26, 324, 72, 2579, 8, 119, 11, 6, 1874, 4, 35, 12, 636, 117, 4, 6, 1264, 585, 47, 4, 6, 1600, 25, 122, 18, 11, 456, 6, 4507, 4, 6, 36, 25, 517, 1202, 90, 11, 6, 2051, 4, 13, 660, 1121, 11, 6, 1874, 4, 384, 2579, 12, 176, 11, 6, 2356, 4, 2930, 3581, 287, 13, 19, 579, 11, 4798, 4, 13, 784, 2567, 12, 6, 2778, 119, 90, 11, 222, 1991, 4, 2138, 10, 503, 5, 215, 22, 2763, 3386, 1159, 240, 299, 196, 196, 16, 4811, 5791, 451, 7, 94, 4667, 4, 4808, 1025, 10, 967, 2338, 1729, 2422,

In [63]:
def norm(s: str) -> str:
    """Lowercases the text and splits it into word and punctuation tokens."""
    return " ".join(re.findall(r"\w+|[^\w\s]", s.lower()))


bleu = BLEUScore(n_gram=4, smooth=True)

for _, row in val_df.iterrows():
    hyp = generate_summary(row["text"], model, sp)
    bleu.update([norm(hyp)], [[norm(row["summary"])]])

print(f"{Fore.MAGENTA}BLEU score:{Style.RESET_ALL} {bleu.compute().item()}")

KeyboardInterrupt: 